In [14]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise FileNotFoundError(f"Could not find data/raw/DHS_Districts from cwd {_cwd}.")

MICRO_DIR = REPO_ROOT / "data" / "raw" / "DHS_microdata"
CROSSWALK_DIR = REPO_ROOT / "data" / "processed" / "district_crosswalks"

IR4_PATH = MICRO_DIR / "NFHS4_2015-16_IndividualRecode" / "IAIR74FL.DTA"
IR5_PATH = MICRO_DIR / "NFHS5_2019-21_IndividualRecode" / "IAIR7EFL.DTA"
# BR files should share the same phase/version code as their IR counterparts
# (74 / 7E). Verify these two paths against your actual folder names first --
# I haven't confirmed your BR folder naming convention.
BR4_PATH = MICRO_DIR / "NFHS4_2015-16_BirthsRecode" / "IABR74FL.DTA"
BR5_PATH = MICRO_DIR / "NFHS5_2019-21_BirthsRecode" / "IABR7EFL.DTA"

for p in [IR4_PATH, IR5_PATH, BR4_PATH, BR5_PATH]:
    print(p, "->", p.exists())

/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS4_2015-16_IndividualRecode/IAIR74FL.DTA -> True
/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS5_2019-21_IndividualRecode/IAIR7EFL.DTA -> True
/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS4_2015-16_BirthsRecode/IABR74FL.DTA -> True
/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS5_2019-21_BirthsRecode/IABR7EFL.DTA -> True


In [15]:
def peek(path, usecols=None, row_limit=5):
    df, meta = pyreadstat.read_dta(str(path), usecols=usecols, row_limit=row_limit)
    return df, meta

_, ir4_meta = peek(IR4_PATH)
_, ir5_meta = peek(IR5_PATH)
_, br4_meta = peek(BR4_PATH)
_, br5_meta = peek(BR5_PATH)

ir4_cols, ir5_cols = set(ir4_meta.column_names), set(ir5_meta.column_names)
br4_cols, br5_cols = set(br4_meta.column_names), set(br5_meta.column_names)

print("sdistri in BR4 directly:", "sdistri" in br4_cols)
print("sdist in BR5 directly:", "sdist" in br5_cols)
print("v104 (duration of residence) in IR4/IR5:", "v104" in ir4_cols, "v104" in ir5_cols)

# Confirm b5 coding before trusting the mortality diagnostic below
_, b5_meta = peek(BR4_PATH, usecols=["b5"])
print("b5 value labels:", b5_meta.variable_value_labels.get("b5"))

sdistri in BR4 directly: True
sdist in BR5 directly: True
v104 (duration of residence) in IR4/IR5: True True
b5 value labels: {0: 'no', 1: 'yes'}


In [16]:
IR4_VARS = [c for c in ["caseid", "sdistri", "v005", "v008", "v025", "v106", "v190"] if c in ir4_cols]
IR5_VARS = [c for c in ["caseid", "sdist",   "v005", "v008", "v025", "v106", "v190"] if c in ir5_cols]

ir4, _ = pyreadstat.read_dta(str(IR4_PATH), usecols=IR4_VARS, apply_value_formats=False)
ir5, _ = pyreadstat.read_dta(str(IR5_PATH), usecols=IR5_VARS, apply_value_formats=False)

ir4 = ir4.rename(columns={"sdistri": "district_code_raw"})
ir5 = ir5.rename(columns={"sdist": "district_code_raw"})
print(ir4.shape, ir5.shape)

(699686, 7) (724115, 7)


In [17]:
crosswalk = pd.read_csv(CROSSWALK_DIR / "nfhs4_nfhs5_stable_district_crosswalk.csv")
print(crosswalk.columns.tolist())

nfhs4_to_stable = dict(zip(crosswalk["district_code"], crosswalk["district_code"]))
nfhs5_to_stable = dict(zip(crosswalk["district_code_nfhs5"], crosswalk["district_code"]))

size_by_district = (
    ir4[ir4["district_code_raw"].isin(nfhs4_to_stable)]
    .groupby("district_code_raw").size()
    .sort_values()
)

n_smallest = size_by_district.head(5).index.tolist()
n_largest = size_by_district.tail(5).index.tolist()
remaining = size_by_district.index.difference(n_smallest + n_largest)
rng = np.random.default_rng(42)
n_random = rng.choice(remaining, size=10, replace=False).tolist()

pilot_districts = sorted(set(n_smallest + n_largest + n_random))
print(f"{len(pilot_districts)} pilot districts: {pilot_districts}")

crosswalk[crosswalk["district_code"].isin(pilot_districts)][
    ["district_code", "district_name_nfhs4", "district_code_nfhs5", "district_name_nfhs5"]
]

['district_code', 'district_name_nfhs4', 'district_name_nfhs5', 'status', 'district_code_nfhs5']
20 pilot districts: [25, 52, 53, 57, 127, 132, 138, 147, 150, 264, 272, 275, 413, 435, 444, 493, 495, 519, 556, 599]


,district_code,district_name_nfhs4,district_code_nfhs5,district_name_nfhs5
24,25,lahul and spiti,25,lahul & spiti
49,52,sahibzada ajit singh nagar,52,sahibzada ajit singh nagar
50,53,shahid bhagat singh nagar,39,shahid bhagat singh nagar
54,57,chamoli,57,chamoli
114,127,kota,127,kota
119,132,saharanpur,132,saharanpur
123,138,meerut,138,meerut
131,147,firozabad,147,firozabad
133,150,bareilly,150,bareilly
240,264,wokha,264,wokha


In [18]:
ir4["stable_district_id"] = ir4["district_code_raw"].map(nfhs4_to_stable)
ir5["stable_district_id"] = ir5["district_code_raw"].map(nfhs5_to_stable)

ir4_pilot = ir4[ir4["stable_district_id"].isin(pilot_districts)].copy()
ir5_pilot = ir5[ir5["stable_district_id"].isin(pilot_districts)].copy()
print(len(ir4_pilot), len(ir5_pilot))

28760 19154


In [19]:
BR4_VARS = [c for c in ["caseid", "bidx", "b3", "b5", "b7"] if c in br4_cols]
BR5_VARS = [c for c in ["caseid", "bidx", "b3", "b5", "b7"] if c in br5_cols]

br4, _ = pyreadstat.read_dta(str(BR4_PATH), usecols=BR4_VARS, apply_value_formats=False)
br5, _ = pyreadstat.read_dta(str(BR5_PATH), usecols=BR5_VARS, apply_value_formats=False)

br4 = br4.merge(ir4_pilot[["caseid", "stable_district_id", "v005", "v008", "v025", "v106", "v190"]],
                 on="caseid", how="inner").assign(round="NFHS-4")
br5 = br5.merge(ir5_pilot[["caseid", "stable_district_id", "v005", "v008", "v025", "v106", "v190"]],
                 on="caseid", how="inner").assign(round="NFHS-5")

births = pd.concat([br4, br5], ignore_index=True)
print(births.shape)
births.head()

(85237, 12)


,caseid,bidx,b3,b5,b7,stable_district_id,v005,v008,v025,v106,v190,round
0,07002601 02,1,1336,1,NaN,413.0,408222,1398,2,3,5,NFHS-4
1,07002603 02,1,1067,1,NaN,413.0,408222,1398,2,1,2,NFHS-4
2,07002610 03,1,1254,1,NaN,413.0,408222,1398,2,0,4,NFHS-4
3,07002610 03,2,1234,1,NaN,413.0,408222,1398,2,0,4,NFHS-4
4,07002610 03,3,1201,1,NaN,413.0,408222,1398,2,0,4,NFHS-4


In [20]:
births["birth_year"] = 1900 + (births["b3"] - 1) // 12
births["birth_month"] = (births["b3"] - 1) % 12 + 1
births["years_before_survey"] = ((births["v008"] - births["b3"]) / 12).apply(np.floor)
births["weight"] = births["v005"] / 1_000_000

births[["round", "stable_district_id", "birth_year", "birth_month", "years_before_survey"]].describe(include="all")

,round,stable_district_id,birth_year,birth_month,years_before_survey
count,85237,85237.000000,85237.000000,85237.000000,85237.000000
unique,2,NaN,NaN,NaN,NaN
top,NFHS-4,NaN,NaN,NaN,NaN
freq,53214,NaN,NaN,NaN,NaN
mean,NaN,243.907141,2004.234863,6.499818,12.483722
std,NaN,164.893100,7.976739,3.402796,7.748652
min,NaN,25.000000,1975.000000,1.000000,0.000000
25%,NaN,132.000000,1998.000000,4.000000,6.000000
50%,NaN,150.000000,2005.000000,7.000000,12.000000
75%,NaN,435.000000,2011.000000,9.000000,18.000000


In [21]:
heaping = (
    births.groupby(["round", "birth_year"])["birth_month"]
    .apply(lambda s: (s == 6).mean())
    .rename("share_month_is_june")
    .reset_index()
)
heaping["flag_heaping"] = heaping["share_month_is_june"] > 0.20
print(heaping[heaping["flag_heaping"]].to_string(index=False))

Empty DataFrame
Columns: [round, birth_year, share_month_is_june, flag_heaping]
Index: []


In [22]:
cell_counts = (
    births.groupby(["round", "stable_district_id", "birth_year"])
    .agg(n_births=("b3", "size"), weighted_births=("weight", "sum"))
    .reset_index()
)
MIN_CELL_N = 30
cell_counts["below_threshold"] = cell_counts["n_births"] < MIN_CELL_N
print(f"{cell_counts['below_threshold'].mean():.1%} of district-year cells have fewer than {MIN_CELL_N} raw births")

by_years_back = births.groupby(["round", "years_before_survey"]).size().rename("n_births").reset_index()
by_years_back.pivot(index="years_before_survey", columns="round", values="n_births").head(25)

29.5% of district-year cells have fewer than 30 raw births


round,NFHS-4,NFHS-5
years_before_survey,,
0.0,1962.0,1104.0
1.0,2087.0,1101.0
2.0,2050.0,1124.0
3.0,2125.0,1152.0
4.0,2156.0,1242.0
5.0,2101.0,1273.0
6.0,2240.0,1299.0
7.0,2188.0,1361.0
8.0,2234.0,1343.0


In [23]:
mortality_by_year = (
    births.groupby(["round", "years_before_survey"])
    .agg(n=("b5", "size"), share_dead=("b5", lambda s: (s == 0).mean()))
    .reset_index()
)
mortality_by_year.pivot(index="years_before_survey", columns="round", values="share_dead").head(25)

round,NFHS-4,NFHS-5
years_before_survey,,
0.0,0.040265,0.035326
1.0,0.040249,0.043597
2.0,0.043902,0.041815
3.0,0.058353,0.037326
4.0,0.051484,0.033011
5.0,0.059020,0.040848
6.0,0.060268,0.043880
7.0,0.066271,0.038207
8.0,0.060877,0.051378


In [24]:
if "v104" in ir4_cols and "v104" in ir5_cols:
    dur4, _ = pyreadstat.read_dta(str(IR4_PATH), usecols=["caseid", "v104"], apply_value_formats=False)
    dur5, _ = pyreadstat.read_dta(str(IR5_PATH), usecols=["caseid", "v104"], apply_value_formats=False)
    dur = pd.concat([dur4.assign(round="NFHS-4"), dur5.assign(round="NFHS-5")], ignore_index=True)

    births_dur = births.merge(dur, on=["caseid", "round"], how="left")
    births_dur["possibly_misattributed"] = births_dur["years_before_survey"] > births_dur["v104"]
    print(births_dur.groupby(["round", "years_before_survey"])["possibly_misattributed"].mean().head(25))
else:
    print("v104 not found -- migration misattribution stays an unquantified limitation for now.")

round   years_before_survey
NFHS-4  0.0                    0.000000
        1.0                    0.034499
        2.0                    0.065854
        3.0                    0.074824
        4.0                    0.096011
        5.0                    0.106616
        6.0                    0.135268
        7.0                    0.132541
        8.0                    0.138765
        9.0                    0.156145
        10.0                   0.156729
        11.0                   0.177271
        12.0                   0.163421
        13.0                   0.171776
        14.0                   0.190879
        15.0                   0.180298
        16.0                   0.212319
        17.0                   0.218256
        18.0                   0.218910
        19.0                   0.231265
        20.0                   0.222579
        21.0                   0.272284
        22.0                   0.260812
        23.0                   0.252000
        24.0

In [25]:
summary = (
    cell_counts.groupby(["round", "birth_year"])
    .agg(total_births=("n_births", "sum"), pct_cells_below_threshold=("below_threshold", "mean"))
    .reset_index()
)
summary.sort_values(["round", "birth_year"], ascending=[True, False])

,round,birth_year,total_births,pct_cells_below_threshold
40,NFHS-4,2016,486,0.692308
39,NFHS-4,2015,1586,0.300000
38,NFHS-4,2014,2098,0.050000
37,NFHS-4,2013,2146,0.000000
36,NFHS-4,2012,2117,0.100000
...,...,...,...,...
45,NFHS-5,1988,66,1.000000
44,NFHS-5,1987,41,1.000000
43,NFHS-5,1986,17,1.000000
42,NFHS-5,1985,7,1.000000
